# A2C Agent Testing Guide on MountainCar-v0

This notebook provides a minimal workflow for testing an **Advantage Actor-Critic (A2C)** agent. It creates a vector environment, trains the agent, and prints the metrics returned by `learn()`. The cells are intended to be edited and rerun while experimenting with the agent and its hyperparameters.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys

import gymnasium as gym
import numpy as np
import torch

POLICY_GRADIENT_DIR = Path.cwd()
if POLICY_GRADIENT_DIR.name != 'policy-gradient':
    POLICY_GRADIENT_DIR = Path('/home/dangineer/Learnspaces/coursera/reinforcement-learning-specialization/rl-lab/deep-rl/policy-gradient')
DEEP_RL_DIR = POLICY_GRADIENT_DIR.parent
if str(DEEP_RL_DIR) not in sys.path:
    sys.path.insert(0, str(DEEP_RL_DIR))
if str(POLICY_GRADIENT_DIR) not in sys.path:
    sys.path.insert(0, str(POLICY_GRADIENT_DIR))

from a2c import A2CAgent

## 1. Configure the Experiment

Adjust these values before running the training cell. Start with a small `MAX_STEPS` while debugging, then increase it for longer experiments.

In [ ]:
ENV_ID = 'MountainCar-v0'
NUM_ENVS = 16
SEED = 2026
MAX_STEPS = 200_000
N_STEPS = 10
LEARNING_RATE = 3e-4
GAMMA = 0.99
ENTROPY_BETA = 0.001
MAX_GRAD_NORM = 0.5

print(f'Environment: {ENV_ID}')
print(f'Number of parallel environments: {NUM_ENVS}')
print(f'Seed: {SEED}')
print(f'Total steps: {MAX_STEPS}')
print(f'Rollout length (n_steps): {N_STEPS}')
print(f'Learning rate: {LEARNING_RATE}')
print(f'Gamma: {GAMMA}')

## 2. Create the Vector Environment and Test the Agent

This cell creates parallel environments, initializes the A2C agent, runs training, and stores the returned metrics in `metrics`. Rerun this cell after changing the configuration above.

In [ ]:
def make_env(env_id, seed, idx):
    def thunk():
        env = gym.make(env_id)
        env.reset(seed=seed + idx)
        env.action_space.seed(seed + idx)
        return env
    return thunk

# Create parallel vector environments
envs = gym.vector.SyncVectorEnv([make_env(ENV_ID, SEED, i) for i in range(NUM_ENVS)])

state_dim = envs.single_observation_space.shape[0]
num_actions = envs.single_action_space.n

agent = A2CAgent(
    state_dim=state_dim,
    num_actions=num_actions,
    lr=LEARNING_RATE,
    gamma=GAMMA,
    entropy_beta=ENTROPY_BETA,
    n_steps=N_STEPS,
    seed=SEED,
    max_grad_norm=MAX_GRAD_NORM,
)

print(f'State dim: {state_dim}, Action dim: {num_actions}')
print(f'Starting training for {MAX_STEPS} steps...')
metrics = agent.learn(envs, max_steps=MAX_STEPS)
print('Training metrics:')
for name, value in metrics.items():
    print(f'  {name}: {value}')
agent.close()
envs.close()
print('Training complete!')

## 3. Inspect the Training Metrics

`A2CAgent.learn()` returns a dictionary containing aggregate information from the training run. The next cell prints a short explanation for each metric.

In [ ]:
metric_descriptions = {
    'actor_loss': 'Policy objective; lower is not always better by itself.',
    'critic_loss': 'Value-function prediction error.',
    'entropy': 'Policy uncertainty and exploration level.',
    'total_loss': 'Combined actor, critic, and entropy objective.',
    'episode_return': 'Average return of completed episodes.',
    'episode_length': 'Average length of completed episodes.',
    'total_steps': 'Number of environment steps collected.',
}

for name, description in metric_descriptions.items():
    print(f'{name}: {metrics[name]}')
    print(f'  {description}')

## 4. Hyperparameter Tuning

A2C is a more advanced version of Actor-Critic method, which is added some techniques to make the training process more stable and efficient. These techniques are:
- Instead of using 1-step td in the update step, A2C use n-steps bootstrapping when compute the TD target
- Using multiple worker running on independent envs, then using batch-based data to update the agent
- Often adding entropy regularization to control and encourage exploration

So the hyperparameters we need to choose carefully for this agent is quite similar to Classic Actor-Critic method, including:
- The learning rate for actor and critic. And, actor learning rate is often smaller than critic lr -> because we want the critic to learn the value function more quickly to maintain a stable, reliable signal for the policy updates.
- Actor and Critic network, based on the problem we are trying to solve to choose the suitable architectures.
- In A2C, we also need to tune the n steps bootstrapping value, which is the number of step taken in the environment before computing the td target
- The number of environment is also a value that worth choose carefully, it not only depends on the algorithm we choose but also our compute resources